# EcoAlert YOLO26n custom waste detector — Colab GPU training

This notebook trains the six-class EcoAlert V1 detector on Google Colab GPU. It intentionally contains no saved outputs or fabricated metrics. Select **Runtime → Change runtime type → GPU** before running. Do not run this notebook on the EcoAlert Windows development machine.

In [ ]:
# Verify that Colab assigned a GPU before installing/training anything.
!nvidia-smi

In [ ]:
# Match backend/vision-service/requirements.txt exactly.
%pip install -q ultralytics==8.4.102

In [ ]:
import torch
import ultralytics

EXPECTED_ULTRALYTICS = "8.4.102"
assert ultralytics.__version__ == EXPECTED_ULTRALYTICS, (
    f"Expected Ultralytics {EXPECTED_ULTRALYTICS}, got {ultralytics.__version__}"
)
assert torch.cuda.is_available(), "GPU is unavailable. Stop and select a Colab GPU runtime."
print({"ultralytics": ultralytics.__version__, "gpu": torch.cuda.get_device_name(0)})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Google Drive layout

Upload the prepared `ecoalert-waste` directory under `MyDrive/EcoAlert/datasets/`. Copy `backend/vision-service/training/data.yaml` into that dataset root. Training runs and weights are written directly to Drive.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import yaml

DRIVE_ROOT = Path('/content/drive/MyDrive/EcoAlert')
DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'ecoalert-waste'
DATA_YAML = DATASET_ROOT / 'data.yaml'
RUNS_ROOT = DRIVE_ROOT / 'training-runs'
RUN_NAME = f"ecoalert-waste-yolo26n-v1-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print({"dataset": str(DATASET_ROOT), "runs": str(RUNS_ROOT), "runName": RUN_NAME})

In [ ]:
# Validate folders, the exact class contract, image/label pairing, and YOLO rows.
EXPECTED_NAMES = {
    0: 'plastic_bottle',
    1: 'plastic_bag',
    2: 'plastic_cup',
    3: 'metal_can',
    4: 'cardboard',
    5: 'glass_bottle',
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}

assert DATA_YAML.is_file(), f"Missing {DATA_YAML}"
with DATA_YAML.open('r', encoding='utf-8') as stream:
    dataset_config = yaml.safe_load(stream)
actual_names = {int(key): value for key, value in dataset_config.get('names', {}).items()}
assert actual_names == EXPECTED_NAMES, f"Class mapping mismatch: {actual_names}"

split_counts = {}
for split in ('train', 'val', 'test'):
    image_dir = DATASET_ROOT / 'images' / split
    label_dir = DATASET_ROOT / 'labels' / split
    assert image_dir.is_dir(), f"Missing directory: {image_dir}"
    assert label_dir.is_dir(), f"Missing directory: {label_dir}"
    images = sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    assert images, f"No images found in {image_dir}"
    image_stems = [path.stem for path in images]
    assert len(image_stems) == len(set(image_stems)), f"Duplicate image stems in {split}"
    missing_labels = [stem for stem in image_stems if not (label_dir / f'{stem}.txt').is_file()]
    assert not missing_labels, (
        f"{split} has images without reviewed label files (use empty files for negatives): "
        f"{missing_labels[:10]}"
    )
    orphan_labels = [path.name for path in label_dir.glob('*.txt') if path.stem not in set(image_stems)]
    assert not orphan_labels, f"{split} has orphan labels: {orphan_labels[:10]}"
    object_count = 0
    for label_path in label_dir.glob('*.txt'):
        for line_number, raw_line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
            line = raw_line.strip()
            if not line:
                continue
            fields = line.split()
            assert len(fields) == 5, f"{label_path}:{line_number} must contain 5 values"
            class_id = int(fields[0])
            coords = [float(value) for value in fields[1:]]
            assert class_id in EXPECTED_NAMES, f"Invalid class {class_id} at {label_path}:{line_number}"
            assert all(0.0 <= value <= 1.0 for value in coords), (
                f"Coordinates outside 0..1 at {label_path}:{line_number}"
            )
            assert coords[2] > 0 and coords[3] > 0, f"Empty box at {label_path}:{line_number}"
            object_count += 1
    split_counts[split] = {"images": len(images), "objects": object_count}

total_images = sum(item['images'] for item in split_counts.values())
ratios = {split: round(item['images'] / total_images, 3) for split, item in split_counts.items()}
print({"splits": split_counts, "ratios": ratios, "recommended": "0.80/0.10/0.10"})

# Use an absolute dataset path in this Colab session without modifying the Drive copy.
runtime_config = dict(dataset_config)
runtime_config['path'] = str(DATASET_ROOT)
RUNTIME_DATA_YAML = Path('/content/ecoalert-waste-data.yaml')
RUNTIME_DATA_YAML.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding='utf-8')
print(f"Validated runtime YAML: {RUNTIME_DATA_YAML}")

## Fine-tune YOLO26n

This starts from the official general-purpose `yolo26n.pt` baseline. The resulting `best.pt` is custom only because it is fine-tuned on the reviewed EcoAlert dataset.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
train_results = model.train(
    data=str(RUNTIME_DATA_YAML),
    imgsz=640,
    epochs=50,
    patience=15,
    device=0,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=False,
    plots=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PT.is_file(), f"Training completed without expected artifact: {BEST_PT}"
print({"runDirectory": str(RUN_DIR), "bestCheckpoint": str(BEST_PT), "bytes": BEST_PT.stat().st_size})

## Held-out test validation

The next cell computes actual values from the completed run. It does not contain target or placeholder metrics. Keep the test split unchanged across release comparisons.

In [ ]:
best_model = YOLO(str(BEST_PT))
test_metrics = best_model.val(
    data=str(RUNTIME_DATA_YAML),
    split='test',
    imgsz=640,
    device=0,
    plots=True,
    project=str(RUNS_ROOT),
    name=f'{RUN_NAME}-test',
)
measured_metrics = {
    'precision': float(test_metrics.box.mp),
    'recall': float(test_metrics.box.mr),
    'mAP50': float(test_metrics.box.map50),
    'mAP50-95': float(test_metrics.box.map),
}
print(measured_metrics)
print({"validationDirectory": str(test_metrics.save_dir), "bestCheckpoint": str(BEST_PT)})

In [ ]:
# Display confusion-matrix artifacts generated by the actual held-out test run.
from IPython.display import display
from PIL import Image

validation_dir = Path(test_metrics.save_dir)
matrix_paths = [
    validation_dir / 'confusion_matrix_normalized.png',
    validation_dir / 'confusion_matrix.png',
]
existing_matrices = [path for path in matrix_paths if path.is_file()]
assert existing_matrices, f"No confusion matrix found in {validation_dir}"
for matrix_path in existing_matrices:
    print(matrix_path.name)
    display(Image.open(matrix_path))

In [ ]:
# Run and display a small set of held-out prediction examples.
test_images = sorted(
    path for path in (DATASET_ROOT / 'images' / 'test').iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)
assert test_images, 'The test split has no images.'
example_results = best_model.predict(
    source=[str(path) for path in test_images[:6]],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project=str(RUNS_ROOT),
    name=f'{RUN_NAME}-prediction-examples',
)
for result in example_results:
    display(Image.fromarray(result.plot()[..., ::-1]))

In [ ]:
# Simple single-image prediction test using best.pt.
TEST_IMAGE = test_images[0]  # Change to another reviewed image if needed.
prediction = best_model.predict(str(TEST_IMAGE), imgsz=640, conf=0.25, device=0, verbose=False)[0]
detections = []
for class_id, confidence in zip(prediction.boxes.cls.tolist(), prediction.boxes.conf.tolist()):
    detections.append({
        'class': prediction.names[int(class_id)],
        'confidence': round(float(confidence), 4),
    })
print({"image": str(TEST_IMAGE), "detections": detections})
display(Image.fromarray(prediction.plot()[..., ::-1]))
print(f"Approved artifact candidate (not yet deployed): {BEST_PT}")

## Next step

Review per-class results, false positives, confusion matrix, and representative predictions. Do not deploy solely from aggregate mAP. If approved, follow `docs/custom-yolo-deployment.md`; keep the existing COCO checkpoint for rollback and do not enable SAM2 as part of this workflow.